# TSE - Tribunal Superior Eleitoral

Example notebook for the TSE scraper.

The TSE jurisprudence portal ([jurisprudencia.tse.jus.br](https://jurisprudencia.tse.jus.br/)) requires a CAPTCHA token, generated in the browser and validated server-side, for every search. The scraper issues plain HTTP requests and cannot produce that token, so **the jurisprudence search is not implemented**: `cjsg` raises `NotImplementedError`.

What the portal serves without a CAPTCHA is covered:

| Method | Description |
|--------|-------------|
| `baixar_inteiro_teor` | Downloads the full-text PDF of a decision, given its `codigoDecisao` |
| `listar_classes` | Case classes of the search form |
| `listar_relatores` | Rapporteurs of the search form |
| `listar_eleicoes` | Election years of the search form |
| `listar_normas` | Legislation of the search form |

In [1]:
import juscraper as jus

tse = jus.scraper("tse")

## Catalogs

Each catalog is one request and comes back as a DataFrame, with the values exactly as the portal returns them.

In [2]:
classes = tse.listar_classes()
print(classes.shape)
classes.head()

(141, 2)


,sigla_classe,descricao_classe
0,RESPE,Recurso Especial Eleitoral
1,AI,Agravo de Instrumento
2,REspEl,RECURSO ESPECIAL ELEITORAL
3,AREspEl,Agravo em Recurso Especial Eleitoral
4,PA,Processo Administrativo


In [3]:
relatores = tse.listar_relatores()
print(relatores.shape)
relatores.head()

(236, 1)


,relator
0,Admar Gonzaga
1,Affonso Augusto Moreira Penna Júnior
2,Affonso Celso
3,Affonso Celso de Assis Figueiredo Júnior
4,Afrânio Antônio Da Costa


In [4]:
tse.listar_eleicoes()["eleicao"].tolist()

['2026',
 '2024',
 '2022',
 '2020',
 '2018',
 '2017',
 '2016',
 '2014',
 '2012',
 '2010',
 '2008',
 '2006',
 '2004',
 '2002',
 '2000',
 '1998',
 '1996',
 '1994',
 '1992',
 '1990',
 '1950']

In [5]:
normas = tse.listar_normas()
print(normas.shape)
normas[["sigla", "tipo", "numero", "ano"]].head()

(75, 5)


,sigla,tipo,numero,ano
0,EC117,EMENDA CONSTITUCIONAL,117,2022
1,ADCT,ATO DAS DISPOSIÇÕES CONSTITUCIONAIS TRANSITÓRIAS,1988,1988
2,CLT,DECRETO-LEI,5452,1943
3,CFD,CONSTITUIÇÃO FEDERAL,1988,1988
4,CADH,TRATADO,0,1969


## Full text of a decision

`baixar_inteiro_teor` takes the decision code (`codigoDecisao`), which is the last segment of the full-text link shown by the portal (`.../rest/download/pdf/<code>`), and returns the path of the saved PDF. The file goes to `<diretorio>/inteiro_teor/<code>.pdf`; without `diretorio`, to the scraper's `download_path`.

The PDF is saved as the court publishes it. Text extraction is left to the caller.

In [6]:
import tempfile
from pathlib import Path

pasta = tempfile.mkdtemp()
caminho = Path(tse.baixar_inteiro_teor(2972172, diretorio=pasta))

print(caminho.relative_to(pasta), caminho.stat().st_size, "bytes")
print(caminho.read_bytes()[:8])

inteiro_teor/2972172.pdf 38777 bytes
b'%PDF-1.7'


### Request spacing

The full-text service refuses requests that come too close together (it accepted one every 10 s and refused at 5 s, measured on 16 November 2026). The scraper therefore waits `sleep_time` seconds (default 10) between consecutive `baixar_inteiro_teor` calls on the same instance, so the cell below takes about 10 s. Change it with `jus.scraper("tse", sleep_time=...)`.

If the service still refuses, the call raises `LimiteDeRequisicoesError` and nothing is saved; wait and call again. A code that does not exist raises `InteiroTeorIndisponivelError`. Both live in `juscraper.courts.tse.download`.

In [7]:
from juscraper.courts.tse.download import InteiroTeorIndisponivelError

try:
    tse.baixar_inteiro_teor(1, diretorio=pasta)
except InteiroTeorIndisponivelError as erro:
    print(type(erro).__name__, "-", erro)

InteiroTeorIndisponivelError - Decisão 1: o TSE respondeu 'text/plain;charset=UTF-8' sem PDF ('Erro ao tentar recuperar PDF: Falha ao recuperar arquivo do SitDoc.'); nada foi gravado.
